[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunilmogadati/systems-in-production/blob/main/implementation/notebooks/Sequence_Models_From_Scratch.ipynb)

# Sequence Models From Scratch — BPTT by Hand

**Companion notebook to** [`playbooks/ai/sequence-models/02_Concepts.md`](https://github.com/sunilmogadati/systems-in-production/blob/main/playbooks/ai/sequence-models/02_Concepts.md).

This notebook trains a tiny vanilla RNN **using only NumPy** — no autograd, no high-level libraries. Every weight, every gradient, every BPTT (Backpropagation Through Time) step is computed by hand.

**Why this matters.** When you write `nn.LSTM(...)` and `loss.backward()` in PyTorch, the framework runs exactly what is in this notebook — just on bigger networks with more sophisticated gates. The unique mechanic of recurrent training is **weight sharing across time** and **gradient accumulation across time**. Once that makes sense, the rest follows.

## What you will see
1. Build a 1×2×1 vanilla RNN (1 input, 2 hidden, 1 output) by hand
2. Forward through 3 timesteps — every number computed
3. Loss at the final timestep
4. **BPTT** — walk backward through time, accumulating gradients into shared weights
5. **Verify** that PyTorch autograd produces the same numbers
6. Train for 200 steps — watch the loss converge
7. Demonstrate **vanishing gradients** with a longer sequence

By the end, `loss.backward()` on an RNN will stop being magic.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
print("NumPy:", np.__version__)

## 1. The Architecture

```
Input dim D = 1, Hidden dim H = 2, Output dim C = 1
Sequence: x_1 = 1.0, x_2 = 0.5, x_3 = -0.5
Target = 0.5 (predict only at t=3)

Forward equations (vanilla RNN):
    z_t   = W_xh · x_t + W_hh · h_{t-1} + b_h
    h_t   = tanh(z_t)
    y_t   = W_hy · h_t + b_y
```

Same `W_xh`, `W_hh`, `b_h` are used at every timestep — **weight sharing across time**.

## 2. Initial Values

In [ ]:
# Inputs
xs = [np.array([1.0]), np.array([0.5]), np.array([-0.5])]
target = 0.5
alpha = 0.1

# Encoder weights (shared across all timesteps)
W_xh = np.array([[0.5], [0.3]], dtype=np.float32)              # shape (H=2, D=1)
W_hh = np.array([[0.1, 0.2], [-0.1, 0.4]], dtype=np.float32)   # shape (H=2, H=2)
b_h  = np.array([0.0, 0.0], dtype=np.float32)                  # shape (H,)

# Output weights (shared across timesteps that produce output, but here we only output at t=T)
W_hy = np.array([[0.3, 0.5]], dtype=np.float32)                # shape (C=1, H=2)
b_y  = np.array([0.0], dtype=np.float32)                       # shape (C,)

print("W_xh:", W_xh.flatten())
print("W_hh:")
print(W_hh)
print("W_hy:", W_hy.flatten())
print(f"target: {target}, learning rate: {alpha}")

## 3. Forward Pass — Through Time

We unroll the network across 3 timesteps. The same weights are used at each step. The hidden state `h_t` carries information forward.

In [ ]:
h = np.zeros(2, dtype=np.float32)    # h_0 = [0, 0]

# Save intermediates for the backward pass
hs = [h.copy()]   # hs[0] = h_0
zs = []
ys = []

print("=== FORWARD ===")
for t, x in enumerate(xs):
    z = W_xh @ x + W_hh @ h + b_h
    h = np.tanh(z)
    y = W_hy @ h + b_y

    zs.append(z.copy())
    hs.append(h.copy())
    ys.append(y.copy())

    print(f"t={t+1}: z={z}, h={h}, y={y}")

y_T = ys[-1][0]    # final output
print(f"\nFinal output y_3 = {y_T:.4f}")
print(f"Target           = {target}")

## 4. Loss

In [ ]:
L = 0.5 * (y_T - target)**2
print(f"L = 0.5 · ({y_T:.4f} - {target})² = {L:.4f}")

## 5. BPTT Backward Pass — Walk Backward Through Time

This is the unique mechanic of recurrent training:

1. Compute output-layer gradients
2. Pass gradient back to the final hidden state `h_T`
3. **Walk backward through time**: at each timestep, compute `∂L/∂z_t`, accumulate gradients into the shared weights `W_xh`, `W_hh`, `b_h`, then propagate `∂L/∂h_{t-1}` to the previous timestep
4. After all timesteps, the shared-weight gradients are the **sum** of contributions from every timestep

This is exactly analogous to CNN's weight-sharing across spatial positions, but here the sharing is across **time**.

### 5.1 Output Layer Gradients (only at final timestep)

In [ ]:
dL_dy_T = y_T - target
print(f"∂L/∂y_T = {dL_dy_T:.4f}")

# Output weights only see the final hidden state
dL_dW_hy = np.outer([dL_dy_T], hs[-1])
dL_db_y  = np.array([dL_dy_T])

print(f"∂L/∂W_hy = {dL_dW_hy.flatten()}")
print(f"∂L/∂b_y  = {dL_db_y}")

### 5.2 Pass Gradient Back to h_T

In [ ]:
# ∂L/∂h_T = W_hyᵀ · ∂L/∂y_T
dL_dh = W_hy.T @ np.array([dL_dy_T])
print(f"∂L/∂h_T (entering recurrence) = {dL_dh}")

### 5.3 Walk Backward Through Time

**This is the BPTT loop.** At each timestep, we compute `∂L/∂z_t`, accumulate gradients into shared weights, and propagate to the previous timestep.

In [ ]:
# Initialize accumulators (zero — we'll add timestep contributions)
dL_dW_xh_total = np.zeros_like(W_xh)
dL_dW_hh_total = np.zeros_like(W_hh)
dL_db_h_total  = np.zeros_like(b_h)

T = len(xs)
print("BPTT walks backward:")
for t in reversed(range(T)):
    # tanh derivative: 1 - h^2
    h_t = hs[t+1]      # hs[0] is h_0, hs[t+1] is h_{t+1}
    tanh_deriv = 1 - h_t**2
    dL_dz = dL_dh * tanh_deriv

    print(f"  t={t+1}: ∂L/∂z_{t+1} = {dL_dz}")

    # Accumulate contributions into shared weights (this is the weight-sharing summation)
    dL_dW_xh_total += np.outer(dL_dz, xs[t])
    dL_dW_hh_total += np.outer(dL_dz, hs[t])    # uses h_{t-1} = hs[t]
    dL_db_h_total  += dL_dz

    # Pass gradient back to previous timestep
    dL_dh = W_hh.T @ dL_dz

print(f"\n∂L/∂W_xh (sum over T timesteps):")
print(dL_dW_xh_total)
print(f"\n∂L/∂W_hh (sum over T timesteps):")
print(dL_dW_hh_total)
print(f"\n∂L/∂b_h (sum over T timesteps): {dL_db_h_total}")

**Notice the accumulation.** Each shared weight got 3 gradient contributions — one per timestep. PyTorch's autograd does exactly this when you call `loss.backward()` on an RNN.

## 6. Verify Against PyTorch Autograd

In [ ]:
import torch

# Convert to torch tensors with grad tracking
W_xh_t = torch.tensor(W_xh, requires_grad=True)
W_hh_t = torch.tensor(W_hh, requires_grad=True)
W_hy_t = torch.tensor(W_hy, requires_grad=True)
b_h_t  = torch.tensor(b_h, requires_grad=True)
b_y_t  = torch.tensor(b_y, requires_grad=True)

# Forward pass — same equations as our manual code
h_t = torch.zeros(2)
for x in xs:
    x_t = torch.tensor(x, dtype=torch.float32)
    z_t = W_xh_t @ x_t + W_hh_t @ h_t + b_h_t
    h_t = torch.tanh(z_t)
y_t = W_hy_t @ h_t + b_y_t
L_t = 0.5 * (y_t[0] - target)**2

# Backward — autograd does BPTT under the hood
L_t.backward()

print("PyTorch ∂L/∂W_xh:")
print(W_xh_t.grad.numpy())
print("Manual:")
print(dL_dW_xh_total)
print(f"Match: {np.allclose(W_xh_t.grad.numpy(), dL_dW_xh_total)}")
print()
print("PyTorch ∂L/∂W_hh:")
print(W_hh_t.grad.numpy())
print("Manual:")
print(dL_dW_hh_total)
print(f"Match: {np.allclose(W_hh_t.grad.numpy(), dL_dW_hh_total)}")
print()
print(f"PyTorch ∂L/∂b_h: {b_h_t.grad.numpy()}")
print(f"Manual         : {dL_db_h_total}")
print(f"Match: {np.allclose(b_h_t.grad.numpy(), dL_db_h_total)}")

**Identical numbers.** PyTorch's autograd is doing the same chain rule + weight-sharing summation through time we just walked through by hand.

## 7. Update Step

In [ ]:
W_xh_new = W_xh - alpha * dL_dW_xh_total
W_hh_new = W_hh - alpha * dL_dW_hh_total
W_hy_new = W_hy - alpha * dL_dW_hy
b_h_new  = b_h  - alpha * dL_db_h_total
b_y_new  = b_y  - alpha * dL_db_y

print("Updated weights:")
print(f"W_xh: {W_xh.flatten()} → {W_xh_new.flatten()}")
print(f"W_hh:")
print(f"  {W_hh}")
print(f"  → {W_hh_new}")
print(f"W_hy: {W_hy.flatten()} → {W_hy_new.flatten()}")

## 8. Train for 200 Steps — Watch the Loss Converge

In [ ]:
# Reset weights
W_xh_t = W_xh.copy()
W_hh_t = W_hh.copy()
W_hy_t = W_hy.copy()
b_h_t  = b_h.copy()
b_y_t  = b_y.copy()

losses = []
EPOCHS = 200

for epoch in range(EPOCHS):
    # Forward
    h = np.zeros(2, dtype=np.float32)
    hs_e = [h.copy()]
    zs_e = []
    for x in xs:
        z = W_xh_t @ x + W_hh_t @ h + b_h_t
        h = np.tanh(z)
        zs_e.append(z.copy())
        hs_e.append(h.copy())
    y_T = (W_hy_t @ h + b_y_t)[0]
    L = 0.5 * (y_T - target)**2
    losses.append(L)

    # Backward
    dL_dy_T = y_T - target
    dL_dW_hy = np.outer([dL_dy_T], hs_e[-1])
    dL_db_y  = np.array([dL_dy_T])
    dL_dh    = W_hy_t.T @ np.array([dL_dy_T])

    dL_dW_xh = np.zeros_like(W_xh_t)
    dL_dW_hh = np.zeros_like(W_hh_t)
    dL_db_h  = np.zeros_like(b_h_t)

    for t in reversed(range(len(xs))):
        tanh_d = 1 - hs_e[t+1]**2
        dL_dz  = dL_dh * tanh_d
        dL_dW_xh += np.outer(dL_dz, xs[t])
        dL_dW_hh += np.outer(dL_dz, hs_e[t])
        dL_db_h  += dL_dz
        dL_dh    = W_hh_t.T @ dL_dz

    # Update
    W_xh_t -= alpha * dL_dW_xh
    W_hh_t -= alpha * dL_dW_hh
    W_hy_t -= alpha * dL_dW_hy
    b_h_t  -= alpha * dL_db_h
    b_y_t  -= alpha * dL_db_y

print(f"Loss step 1:   {losses[0]:.4f}")
print(f"Loss step 50:  {losses[49]:.4f}")
print(f"Loss step 100: {losses[99]:.4f}")
print(f"Loss step 200: {losses[-1]:.6f}")
print()
print(f"Final y_T: {y_T:.4f}, target: {target}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Vanilla RNN training (single sequence) — manual BPTT')
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.show()

## 9. Demonstrate Vanishing Gradients

Vanilla RNN's central weakness: gradients vanish across long sequences. Let's see it.

We'll use a longer sequence (T=20) and track the gradient magnitude as it flows backward through time. The earliest timesteps should get a much smaller gradient than the latest.

In [ ]:
# A longer sequence
T_long = 20
xs_long = [np.random.randn(1).astype(np.float32) for _ in range(T_long)]

# Use the original (untrained) weights
W_xh = np.array([[0.5], [0.3]], dtype=np.float32)
W_hh = np.array([[0.1, 0.2], [-0.1, 0.4]], dtype=np.float32)
b_h  = np.array([0.0, 0.0], dtype=np.float32)
W_hy = np.array([[0.3, 0.5]], dtype=np.float32)
b_y  = np.array([0.0], dtype=np.float32)

# Forward
h = np.zeros(2, dtype=np.float32)
hs_long = [h.copy()]
for x in xs_long:
    z = W_xh @ x + W_hh @ h + b_h
    h = np.tanh(z)
    hs_long.append(h.copy())

# Loss at final timestep
y_T = (W_hy @ h + b_y)[0]
target_long = 0.5

# BPTT and track per-timestep gradient norm
dL_dy_T = y_T - target_long
dL_dh = W_hy.T @ np.array([dL_dy_T])

per_timestep_grad_norm = []
for t in reversed(range(T_long)):
    tanh_d = 1 - hs_long[t+1]**2
    dL_dz = dL_dh * tanh_d
    per_timestep_grad_norm.append(np.linalg.norm(dL_dh))
    dL_dh = W_hh.T @ dL_dz

# Reverse so timestep 0 is first
per_timestep_grad_norm = per_timestep_grad_norm[::-1]

plt.figure(figsize=(9, 4))
plt.plot(range(T_long), per_timestep_grad_norm, marker='o')
plt.xlabel('Timestep (0 = earliest)')
plt.ylabel('||∂L/∂h_t|| (gradient magnitude)')
plt.title('Vanishing gradient: gradient at earliest timesteps shrinks dramatically')
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.show()

print("Gradient at the latest timestep:", per_timestep_grad_norm[-1])
print("Gradient at the earliest timestep:", per_timestep_grad_norm[0])
print(f"Ratio: ~{per_timestep_grad_norm[-1] / max(per_timestep_grad_norm[0], 1e-30):.1e}x smaller at the start")

**This is why vanilla RNNs cannot learn long-range dependencies.** The gradient at the earliest timesteps is many orders of magnitude smaller than at the latest. The model's earliest weights effectively get no learning signal — those time positions are invisible during training.

LSTM solves this by replacing the multiplicative recurrence with an **additive cell-state update**, which preserves gradient flow even across hundreds of timesteps. See the LSTM equations in [02 — Concepts → LSTM](https://github.com/sunilmogadati/systems-in-production/blob/main/playbooks/ai/sequence-models/02_Concepts.md#lstm--the-gated-solution).

## What you just did

| Step | What you proved |
|---|---|
| Forward through time | RNN cell is just `tanh(W·x + W·h + b)`, applied with shared weights at every timestep |
| Loss at final timestep | Standard MSE; nothing recurrent-specific |
| `∂L/∂y_T → ∂L/∂h_T` | Standard backprop through the output layer |
| **BPTT loop** | Walks backward through every timestep, accumulating gradients into shared weights — the unique mechanic |
| **Weight-sharing accumulation** | `∂L/∂W_xh` is the SUM of contributions from every timestep, exactly like CNN's spatial weight-sharing accumulation |
| PyTorch verification | Manual gradients match `loss.backward()` exactly |
| 200-step training | Loss converges; the small RNN learns the target |
| Vanishing gradient demo | Gradient at earliest timesteps is 10⁻¹⁰ smaller than at latest — explains why vanilla RNN cannot learn long-range dependencies |

When you write `nn.LSTM` + `loss.backward()` in PyTorch, you now know exactly what is happening underneath. The LSTM gates are an addition to the same scaffolding you just built by hand — they preserve gradient flow through time the way ResNet preserves it through depth.

---

## What's Next

You have built BPTT by hand. Now build the modern recurrent stack:

- **[Sequence Models → 03 Hello World](https://github.com/sunilmogadati/systems-in-production/blob/main/playbooks/ai/sequence-models/03_Hello_World.md)** — Full LSTM in 60 lines of PyTorch
- **[Sequence Models → 02 Concepts → LSTM](https://github.com/sunilmogadati/systems-in-production/blob/main/playbooks/ai/sequence-models/02_Concepts.md#lstm--the-gated-solution)** — How gating solves vanishing gradients
- **[Sequence Models → 04 How It Works](https://github.com/sunilmogadati/systems-in-production/blob/main/playbooks/ai/sequence-models/04_How_It_Works.md)** — Diagnosing vanishing/exploding gradients in production
- **[Architectures → rnn-lstm.md](https://github.com/sunilmogadati/systems-in-production/blob/main/playbooks/ai/sequence-models/architectures/rnn-lstm.md)** — Single-doc reference covering everything: BPTT, gates, code patterns, math, Q&A

The math you just walked through is the foundation. Everything else — LSTMs, GRUs, attention, Transformers — is built on top of it.